In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.table("healthcare_catalog.silver.hospitals")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df = df.withColumns({
    'rating': col('rating').cast('int'),
    'meets_criteria_for_maternity': when(col('meets_criteria_for_maternity') == 'Y', True).otherwise(False),
})

In [0]:
display(df)

In [0]:
# count statewise hospitals
df.groupBy('state').avg('rating').show()

## “How does healthcare quality vary by state?”
Metrics:
- Total hospitals
- Average rating
- Emergency coverage

In [0]:
fact_state_wise_kpi_df = df.groupBy('state').agg(
    avg('rating').alias('avg_rating'),
    count_distinct('facility_id').alias('total_hospitals'),
    sum(col('has_emergency_service').cast('int')).alias('total_emergency_service_hospital'),
    sum(col('meets_criteria_for_maternity').cast('int')).alias('total_maternity_hospital')
)
display(fact_state_wise_kpi_df.limit(3))


## How many High / Medium / Low hospitals per state?
This is perfect for:
- Bar charts
- Heat maps
- Quality comparisons

In [0]:
fact_state_class_wise_kpi_df = df.groupBy('state', 'class').agg(
  count_distinct('facility_id').alias('total_hospitals'),
  sum(col('has_emergency_service').cast('int')).alias('total_emergency_service_hospital'),
  sum(col('meets_criteria_for_maternity').cast('int')).alias('total_maternity_hospital')
)
display(fact_state_class_wise_kpi_df.limit(3))

## This is a denormalized lookup table.

In [0]:
dim_hospital_df = (
    df
    .select(
        "facility_id",
        "facility_name",
        "hospital_type",
        "hospital_ownership",
        "city",
        "state",
        "county",
        "telephone_number",
        "rating",
        "class",
        "has_emergency_service"
    )
)
display(dim_hospital_df.limit(3))


## Create Gold Schema and save data into gold tables

In [0]:
gold_schema = "healthcare_catalog.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

In [0]:
fact_state_wise_kpi_df.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.fact_state_wise_kpi")
fact_state_class_wise_kpi_df.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.fact_state_class_wise_kpi")
dim_hospital_df.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_hospital")